In [13]:
import pandas as pd
import geopandas as gpd
from pathlib import Path

In [53]:
GTFS_DIR = Path("../../data/raw/transport/gtfs")
stops = pd.read_csv(GTFS_DIR / "stops.txt")
routes = pd.read_csv(GTFS_DIR / "routes.txt")
trips = pd.read_csv(GTFS_DIR / "trips.txt")
stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt")

/var/folders/kc/4swzx7w979z6w9js5c61gt7h0000gn/T/ipykernel_45884/2377360954.py:2: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  stops = pd.read_csv(GTFS_DIR / "stops.txt")
/var/folders/kc/4swzx7w979z6w9js5c61gt7h0000gn/T/ipykernel_45884/2377360954.py:4: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  trips = pd.read_csv(GTFS_DIR / "trips.txt")
/var/folders/kc/4swzx7w979z6w9js5c61gt7h0000gn/T/ipykernel_45884/2377360954.py:5: DtypeWarning: Columns (0,3,5,10) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt")


In [54]:
print(stops.columns)
print(routes.columns)
print(trips.columns)
print(stop_times.columns)

Index(['stop_id', 'stop_code', 'stop_name', 'stop_lat', 'stop_lon',
       'location_type', 'parent_station', 'wheelchair_boarding', 'level_id',
       'platform_code'],
      dtype='object')
Index(['route_id', 'agency_id', 'route_short_name', 'route_long_name',
       'route_desc', 'route_type', 'route_color', 'route_text_color',
       'exact_times'],
      dtype='object')
Index(['route_id', 'service_id', 'trip_id', 'shape_id', 'trip_headsign',
       'direction_id', 'block_id', 'wheelchair_accessible', 'route_direction',
       'trip_note', 'bikes_allowed'],
      dtype='object')
Index(['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence',
       'stop_headsign', 'pickup_type', 'drop_off_type', 'shape_dist_traveled',
       'timepoint', 'stop_note'],
      dtype='object')


In [55]:
routes["route_type"].value_counts(dropna=False)

route_type
712    8642
700    1147
204     159
205     136
714      72
106      27
4        24
2        17
900       6
401       1
Name: count, dtype: int64

In [56]:
rail_route_types = [2, 401]

rail_routes = routes[routes["route_type"].isin(rail_route_types)].copy()
print("Rail/metro routes:", len(rail_routes))
rail_routes[["route_id", "route_type"]].head()

Rail/metro routes: 18


,route_id,route_type
2332,2-BMT-sj2-1,2
2333,2-CCN-sj2-1,2
2334,2-SCO-sj2-1,2
2335,2-T1-N-sj2-1,2
2336,2-T1-W-sj2-1,2


In [57]:
rail_trips = trips[trips["route_id"].isin(rail_routes["route_id"])].copy()
print("Rail/metro trips:", len(rail_trips))

Rail/metro trips: 63810


In [58]:
rail_stop_times = stop_times[stop_times["trip_id"].isin(rail_trips["trip_id"])].copy()
print("Rail/metro stop_times:", len(rail_stop_times))

Rail/metro stop_times: 1002804


In [59]:
used_stop_ids = pd.Series(rail_stop_times["stop_id"].dropna().unique())
print("Used stop_ids:", len(used_stop_ids))
used_stop_ids.head()

Used stop_ids: 832


0     207262
1     207191
2    2070141
3    2069131
4    2067141
dtype: object

In [60]:
used_stops = stops[stops["stop_id"].isin(used_stop_ids)].copy()
print("Used stops rows:", len(used_stops))
used_stops.head()

Used stops rows: 90


,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,level_id,platform_code
15,2000321,2000321.0,"Central Station, Platform 1",-33.883165,151.205523,NaN,200060,1.0,Level 1,1
18,2000324,2000324.0,"Central Station, Platform 4",-33.883317,151.205812,NaN,200060,1.0,Level 1,4
19,2000325,2000325.0,"Central Station, Platform 5",-33.883360,151.205879,NaN,200060,1.0,Level 1,5
20,2000326,2000326.0,"Central Station, Platform 6",-33.883418,151.206011,NaN,200060,1.0,Level 1,6
21,2000327,2000327.0,"Central Station, Platform 7",-33.883452,151.206067,NaN,200060,1.0,Level 1,7


In [61]:
for col in ["stop_id", "stop_name", "stop_lat", "stop_lon", "location_type", "parent_station"]:
    if col in used_stops.columns:
        print(col, used_stops[col].head().tolist())

stop_id ['2000321', '2000324', '2000325', '2000326', '2000327']
stop_name ['Central Station, Platform 1', 'Central Station, Platform 4', 'Central Station, Platform 5', 'Central Station, Platform 6', 'Central Station, Platform 7']
stop_lat [-33.88316452, -33.88331683, -33.88335959, -33.88341813, -33.88345227]
stop_lon [151.20552345, 151.20581181, 151.20587873, 151.20601124, 151.20606693]
location_type [nan, nan, nan, nan, nan]
parent_station ['200060', '200060', '200060', '200060', '200060']


In [62]:
if "location_type" in used_stops.columns:
    print(used_stops["location_type"].value_counts(dropna=False))
else:
    print("No location_type column in used_stops")

location_type
NaN    90
Name: count, dtype: int64


In [63]:
if "parent_station" in used_stops.columns:
    parent_station_ids = (
        used_stops["parent_station"]
        .dropna()
        .astype(str)
        .str.strip()
    )
    parent_station_ids = parent_station_ids[parent_station_ids != ""].unique()
else:
    parent_station_ids = []

print("Parent station ids:", len(parent_station_ids))

Parent station ids: 44


In [64]:
if len(parent_station_ids) > 0:
    parent_stations = stops[stops["stop_id"].astype(str).isin(parent_station_ids)].copy()
else:
    parent_stations = pd.DataFrame(columns=stops.columns)

print("Parent station rows:", len(parent_stations))
parent_stations.head()

Parent station rows: 44


,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,level_id,platform_code
48,200060,NaN,Central Station,-33.884024,151.206203,1.0,NaN,0.0,NaN,NaN
418,216710,NaN,Glenfield Station,-33.972208,150.893067,1.0,NaN,0.0,NaN,NaN
984,229310,NaN,Newcastle Interchange,-32.924041,151.759082,1.0,NaN,0.0,NaN,NaN
1007,230310,NaN,Hamilton Station,-32.918463,151.748498,1.0,NaN,0.0,NaN,NaN
1249,256020,NaN,Campbelltown Station,-34.063556,150.814631,1.0,NaN,0.0,NaN,NaN


In [65]:
if len(parent_stations) > 0 and "location_type" in parent_stations.columns:
    print(parent_stations["location_type"].value_counts(dropna=False))

location_type
1.0    44
Name: count, dtype: int64


In [66]:
if "parent_station" in used_stops.columns:
    no_parent_stops = used_stops[
        used_stops["parent_station"].isna()
        | (used_stops["parent_station"].astype(str).str.strip() == "")
    ].copy()
else:
    no_parent_stops = used_stops.copy()

print("Stops without parent_station:", len(no_parent_stops))

Stops without parent_station: 0


In [67]:
station_candidates = pd.concat(
    [parent_stations, no_parent_stops],
    ignore_index=True
).drop_duplicates(subset=["stop_id"])

print("Station candidate rows:", len(station_candidates))
station_candidates.head()

Station candidate rows: 44


,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,level_id,platform_code
0,200060,NaN,Central Station,-33.884024,151.206203,1.0,NaN,0.0,NaN,NaN
1,216710,NaN,Glenfield Station,-33.972208,150.893067,1.0,NaN,0.0,NaN,NaN
2,229310,NaN,Newcastle Interchange,-32.924041,151.759082,1.0,NaN,0.0,NaN,NaN
3,230310,NaN,Hamilton Station,-32.918463,151.748498,1.0,NaN,0.0,NaN,NaN
4,256020,NaN,Campbelltown Station,-34.063556,150.814631,1.0,NaN,0.0,NaN,NaN


In [68]:
station_candidates = station_candidates.dropna(subset=["stop_lat", "stop_lon"]).copy()
print("Station candidates with coords:", len(station_candidates))

Station candidates with coords: 44


In [69]:
rail_metro_stations_gdf = gpd.GeoDataFrame(
    station_candidates.copy(),
    geometry=gpd.points_from_xy(station_candidates["stop_lon"], station_candidates["stop_lat"]),
    crs="EPSG:4326"
)

rail_metro_stations_gdf.head()

,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,level_id,platform_code,geometry
0,200060,NaN,Central Station,-33.884024,151.206203,1.0,NaN,0.0,NaN,NaN,POINT (151.2062 -33.88402)
1,216710,NaN,Glenfield Station,-33.972208,150.893067,1.0,NaN,0.0,NaN,NaN,POINT (150.89307 -33.97221)
2,229310,NaN,Newcastle Interchange,-32.924041,151.759082,1.0,NaN,0.0,NaN,NaN,POINT (151.75908 -32.92404)
3,230310,NaN,Hamilton Station,-32.918463,151.748498,1.0,NaN,0.0,NaN,NaN,POINT (151.7485 -32.91846)
4,256020,NaN,Campbelltown Station,-34.063556,150.814631,1.0,NaN,0.0,NaN,NaN,POINT (150.81463 -34.06356)


In [70]:
print("Final rail+metro station rows:", len(rail_metro_stations_gdf))
print(rail_metro_stations_gdf.columns)

Final rail+metro station rows: 44
Index(['stop_id', 'stop_code', 'stop_name', 'stop_lat', 'stop_lon',
       'location_type', 'parent_station', 'wheelchair_boarding', 'level_id',
       'platform_code', 'geometry'],
      dtype='object')


In [71]:
if "location_type" in rail_metro_stations_gdf.columns:
    print(rail_metro_stations_gdf["location_type"].value_counts(dropna=False))

location_type
1.0    44
Name: count, dtype: int64


In [72]:
obj_cols = rail_metro_stations_gdf.select_dtypes(include=["object"]).columns
rail_metro_stations_gdf[obj_cols] = rail_metro_stations_gdf[obj_cols].astype("string")

In [73]:
print("Final rail+metro station rows:", len(rail_metro_stations_gdf))

Final rail+metro station rows: 44


In [74]:
if "location_type" in rail_metro_stations_gdf.columns:
    print(rail_metro_stations_gdf["location_type"].value_counts(dropna=False))

location_type
1.0    44
Name: count, dtype: int64


In [75]:
print(routes[["route_id", "route_type", "route_short_name", "route_long_name"]].sample(50, random_state=42))

             route_id  route_type route_short_name  \
6930   58-S85-2-sj2-1         712             S852   
518    11-653-0-sj2-1         712             6530   
8813   72-S95-6-sj2-1         712             S956   
8701   72-S50-7-sj2-1         712             S507   
6978     59-273-sj2-1         700              273   
7458   60-S48-0-sj2-1         712             S480   
8141   71-S32-3-sj2-1         712             S323   
2748   26-280-1-sj2-1         712             2801   
2816      27-41-sj2-1         700               41   
3614   31-S12-0-sj2-1         712             S120   
3037   27-S84-1-sj2-1         712             S841   
4262   42-S93-1-sj2-1         712             S931   
33      1-71S-C-sj2-4         714             71SC   
3728   31-S37-2-sj2-1         712             S372   
5718   54-S65-9-sj2-1         712             S659   
10102  77-S56-3-sj2-1         712             S563   
718    12-105-0-sj2-1         712             1050   
9313   75-S53-4-sj2-1       

In [76]:
for rt in [2, 401, 700, 712, 714, 900]:
    print("\nroute_type =", rt)
    print(routes.loc[routes["route_type"] == rt, ["route_short_name", "route_long_name"]].head(10))


route_type = 2
     route_short_name                        route_long_name
2332              BMT                    Blue Mountains Line
2333              CCN       Central Coast and Newcastle Line
2334              SCO                       South Coast Line
2335               T1        T1 North Shore and Western Line
2336               T1        T1 North Shore and Western Line
2337               T2      T2 Leppington and Inner West Line
2338               T3       T3 Liverpool and Inner West Line
2339               T4  T4 Eastern Suburbs and Illawarra Line
2340               T5                     T5 Cumberland Line
2341               T6         T6 Lidcombe and Bankstown Line

route_type = 401
     route_short_name                         route_long_name
3295               M1  M1 Metro North West and Bankstown Line

route_type = 700
    route_short_name                                    route_long_name
41              SC01                                 Kiama to Bomaderry
42       

In [77]:
route_text = (
    routes["route_short_name"].fillna("").astype(str) + " " +
    routes["route_long_name"].fillna("").astype(str)
).str.upper()

for kw in ["METRO", "TRAIN", "T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8", "T9", "M1"]:
    print(kw, route_text.str.contains(kw, na=False).sum())

METRO 6
TRAIN 0
T1 3
T2 8
T3 4
T4 12
T5 1
T6 4
T7 4
T8 6
T9 3
M1 1


In [78]:
route_text = (
    routes["route_short_name"].fillna("").astype(str).str.upper() + " " +
    routes["route_long_name"].fillna("").astype(str).str.upper()
)

rail_mask = (
    routes["route_type"].isin([2, 401, 714])
    |
    route_text.str.contains(
        r"\bMETRO\b|\bM1\b|\bT1\b|\bT2\b|\bT3\b|\bT4\b|\bT5\b|\bT6\b|\bT7\b|\bT8\b|\bT9\b|\bBMT\b|\bCCN\b|\bSCO\b|\bSH\b|\bSC\b",
        regex=True,
        na=False,
    )
)

rail_routes = routes[rail_mask].copy()

print("Rail/metro routes:", len(rail_routes))
rail_routes[["route_id", "route_type", "route_short_name", "route_long_name"]].head(50)

Rail/metro routes: 166


,route_id,route_type,route_short_name,route_long_name
0,1-10T-4-sj2-1,714,10T4,"Sutherland, then all stations to Waterfall"
1,1-10T-4-sj2-2,714,10T4,"Sutherland, then all stations to Waterfall"
2,1-11S-H-sj2-1,714,11SH,"Glenfield, Campbelltown, Picton, Mittagong, Bo..."
3,1-14S-C-sj2-1,714,14SC,"Thirroul, then all stations to Dapto"
4,1-14S-C-sj2-2,714,14SC,"Thirroul, then all stations to Dapto"
5,1-19T-3-sj2-1,714,19T3,"Cabramatta, then Lidcombe (Express Service)"
6,1-23T-8-sj2-1,714,23T8,"Sydenham, Wolli Creek, Turrella, then all stat..."
7,1-27S-C-sj2-1,714,27SC,"Wollongong, then all stations to Port Kembla"
8,1-27S-C-sj2-2,714,27SC,"Wollongong, then all stations to Port Kembla"
9,1-27S-C-sj2-3,714,27SC,"Wollongong, then all stations to Port Kembla"


In [79]:
for df, cols in [
    (stops, ["stop_id", "parent_station"]),
    (routes, ["route_id"]),
    (trips, ["route_id", "trip_id"]),
    (stop_times, ["trip_id", "stop_id"]),
]:
    for col in cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()
            df.loc[df[col].isin(["", "nan", "None", "<NA>"]), col] = pd.NA

In [80]:
rail_trips = trips[trips["route_id"].isin(rail_routes["route_id"])].copy()
print("Rail/metro trips:", len(rail_trips))

Rail/metro trips: 68783


In [81]:
rail_stop_times = stop_times[stop_times["trip_id"].isin(rail_trips["trip_id"])].copy()
print("Rail/metro stop_times:", len(rail_stop_times))

Rail/metro stop_times: 1049378


In [82]:
used_stop_ids = rail_stop_times["stop_id"].dropna().astype(str).str.strip().unique()
print("Used stop_ids:", len(used_stop_ids))

Used stop_ids: 2047


In [83]:
used_stops = stops[stops["stop_id"].isin(used_stop_ids)].copy()
print("Used stops rows:", len(used_stops))
used_stops.head()

Used stops rows: 2047


,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,level_id,platform_code
5,2000136,2000136.0,"QVB, York St, Stand E",-33.870119,151.206287,NaN,2000171,0.0,Level 0,NaN
15,2000321,2000321.0,"Central Station, Platform 1",-33.883165,151.205523,NaN,200060,1.0,Level 1,1
16,2000322,2000322.0,"Central Station, Platform 2",-33.883224,151.205634,NaN,200060,1.0,Level 1,2
17,2000323,2000323.0,"Central Station, Platform 3",-33.883258,151.205701,NaN,200060,1.0,Level 1,3
18,2000324,2000324.0,"Central Station, Platform 4",-33.883317,151.205812,NaN,200060,1.0,Level 1,4


In [84]:
parent_station_ids = (
    used_stops["parent_station"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print("Parent station ids:", len(parent_station_ids))

Parent station ids: 1284


In [85]:
parent_stations = stops[stops["stop_id"].isin(parent_station_ids)].copy()
print("Parent station rows:", len(parent_stations))

Parent station rows: 1284


In [86]:
print("Used stop_ids:", len(used_stop_ids))
print("Used stops rows:", len(used_stops))
print("Parent station rows:", len(parent_stations))

Used stop_ids: 2047
Used stops rows: 2047
Parent station rows: 1284


In [87]:
no_parent_stops = used_stops[
    used_stops["parent_station"].isna()
].copy()

print("Stops without parent_station:", len(no_parent_stops))

Stops without parent_station: 0


In [88]:
station_candidates = pd.concat(
    [parent_stations, no_parent_stops],
    ignore_index=True
).drop_duplicates(subset=["stop_id"])

print("Station candidate rows:", len(station_candidates))

Station candidate rows: 1284


In [89]:
station_candidates = station_candidates.dropna(subset=["stop_lat", "stop_lon"]).copy()
print("Station candidates with coords:", len(station_candidates))

Station candidates with coords: 1284


In [90]:
rail_metro_stations_gdf = gpd.GeoDataFrame(
    station_candidates.copy(),
    geometry=gpd.points_from_xy(
        station_candidates["stop_lon"],
        station_candidates["stop_lat"]
    ),
    crs="EPSG:4326"
)

print("Final rail+metro station rows:", len(rail_metro_stations_gdf))
rail_metro_stations_gdf.head()

Final rail+metro station rows: 1284


,stop_id,stop_code,stop_name,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,level_id,platform_code,geometry
0,2000171,NaN,QVB,-33.871692,151.206656,1.0,NaN,0.0,NaN,NaN,POINT (151.20666 -33.87169)
1,200060,NaN,Central Station,-33.884024,151.206203,1.0,NaN,0.0,NaN,NaN,POINT (151.2062 -33.88402)
2,201510,NaN,Redfern Station,-33.892048,151.198419,1.0,NaN,0.0,NaN,NaN,POINT (151.19842 -33.89205)
3,204210,NaN,Newtown Station,-33.897913,151.179377,1.0,NaN,0.0,NaN,NaN,POINT (151.17938 -33.89791)
4,204420,NaN,Sydenham Station,-33.914459,151.166526,1.0,NaN,0.0,NaN,NaN,POINT (151.16653 -33.91446)


In [91]:
obj_cols = rail_metro_stations_gdf.select_dtypes(include=["object"]).columns
rail_metro_stations_gdf[obj_cols] = rail_metro_stations_gdf[obj_cols].astype("string")

In [92]:
Path("../../data/processed/transport").mkdir(parents=True, exist_ok=True)

rail_metro_stations_gdf.to_parquet(
    "../../data/processed/transport/rail_metro_stations_raw.parquet",
    index=False
)